In [0]:
dbutils.fs.ls("dbfs:/Volumes/data/raw/sales/raw_data/retail/dev/")

In [0]:
customers_df = spark.read \
    .format("json") \
    .option("inferSchema", "true") \
    .option("mode", "FAILFAST") \
    .option("columnNameOfCorruptRecord", "_corrupt_record") \
    .load("/Volumes/data/raw/sales/raw_data/retail/dev/customers")

customers_df.printSchema()



In [0]:
from pyspark.sql.functions import col

# display(customers_df.count())
display(customers_df.dropna().count())
# display(customers_df.dropDuplicates(["phone"]).filter(col("phone").isNull()))
# display(customers_df.filter(col("phone").isNotNull()).count())
# display(customers_df.filter(col("phone").isNull()).count())
# display(customers_df.filter(col("phone").isNotNull()).count())
# display(customers_df)

In [0]:
inventory_df = spark.read \
    .format("json") \
    .option("mode", "FAILFAST") \
    .option("columnNameOfCorruptRecord", "_corrupt_record") \
    .load("/Volumes/data/raw/sales/raw_data/retail/dev/inventory")


inventory_df.printSchema()


In [0]:
from pyspark.sql.functions import to_date, col
display(inventory_df.select(to_date(col("last_updated")).alias("date")).distinct())

In [0]:
# unique_inventory_df = inventory_df.distinct()
# display(inventory_df.subtract(unique_inventory_df))

dup_df = inventory_df.select("product_id", "store_id").distinct()

display(dup_df.dropDuplicates(["product_id", "store_id"]))


In [0]:
duplicate_df = inventory_df.groupBy("product_id", "store_id").count().filter(col("count") > 1)
display(duplicate_df)

In [0]:
display(dbutils.fs.ls("/Volumes/data/raw/sales/raw_data/retail/dev/pos_transactions/"))

In [0]:
display(dbutils.fs.ls("/Volumes/data/raw/sales/raw_data"))

In [0]:
spark.read.format("json") \
        .load("/Volumes/data/raw/sales/raw_data/date=2026-03-31/hour=00/").printSchema()


In [0]:
df = spark.read.format("json") \
        .load("/Volumes/data/raw/sales/raw_data/")

In [0]:
display(df.count())

In [0]:
%sql
-- show schemas in sales;
show grants on catalog sales;

In [0]:
%sql
-- drop table retails.bronze.pos_transactions_raw;

In [0]:
%sql

-- CREATE TABLE retails.bronze.customers_raw
-- USING DELTA
-- LOCATION 's3://thoughtbulls-dp-uc-root-ap-south-1-8affd0fb/external_data/bronze/retails/customers_raw';
-------------------------------------------------------------------------------------------------------------

-- CREATE TABLE retails.bronze.inventory_raw
-- USING DELTA
-- LOCATION 's3://thoughtbulls-dp-uc-root-ap-south-1-8affd0fb/external_data/bronze/retails/inventory_raw';
-------------------------------------------------------------------------------------------------------------

-- CREATE TABLE retails.bronze.products_raw
-- USING DELTA
-- LOCATION 's3://thoughtbulls-dp-uc-root-ap-south-1-8affd0fb/external_data/bronze/retails/products_raw';
-------------------------------------------------------------------------------------------------------------

-- CREATE TABLE retails.bronze.stores_raw
-- USING DELTA
-- LOCATION 's3://thoughtbulls-dp-uc-root-ap-south-1-8affd0fb/external_data/bronze/retails/stores_raw';
-------------------------------------------------------------------------------------------------------------

-- CREATE TABLE retails.bronze.pos_transactions_raw
-- USING DELTA
-- LOCATION 's3://thoughtbulls-dp-uc-root-ap-south-1-8affd0fb/external_data/bronze/retails/pos_transactions_raw';


In [0]:
%sql
-- ALTER TABLE retails.bronze.pos_transactions_raw OWNER TO `dp-sales-engineers`;
-- ALTER TABLE retails.bronze.customers_raw OWNER TO `dp-sales-engineers`;
-- ALTER TABLE retails.bronze.inventory_raw OWNER TO `dp-sales-engineers`;
-- ALTER TABLE retails.bronze.products_raw OWNER TO `dp-sales-engineers`;
-- ALTER TABLE retails.bronze.stores_raw OWNER TO `dp-sales-engineers`;

In [0]:
pos_df = spark.read \
    .format("json") \
    .option("mode", "DROPMALFORMED") \
    .option("columnNameOfCorruptRecord", "_corrupt_record") \
    .load("/Volumes/data/raw/sales/raw_data/retail/dev/pos_transactions/")

# display(pos_df);
pos_df.printSchema()

In [0]:
from pyspark.sql.functions import col, to_date
display(pos_df.select(to_date(col("date"))).distinct())

In [0]:
%sql
-- ALTER TABLE retails.bronze.pos_transactions_raw
-- ADD COLUMNS (
--   item_id STRING,
--   new_column STRING,
--   order_id STRING,
--   payment_type STRING,
--   price STRING,
--   qty STRING,
--   store_id STRING,
--   txn_time STRING,
--   date DATE,
--   hour INT,
-- -- Metadata columns
--   ingestion_timestamp TIMESTAMP,
--   load_date DATE,
--   source_file_name STRING,
--   record_id STRING,
--   source_system STRING,
--   is_deleted BOOLEAN,

--   raw_data STRING,
--   _rescued_data STRING
-- );

  


In [0]:
%sql
REPLACE TABLE retails.bronze.pos_transactions_raw
USING DELTA
PARTITIONED BY (load_date)
AS SELECT * FROM retails.bronze.pos_transactions_raw;

In [0]:
%sql
ALTER TABLE retails.bronze.customers_raw
ADD COLUMNS(
  customer_id STRING,
  loyalty_points STRING,
  name STRING,
  phone STRING,
  
-- Metadata columns
  ingestion_timestamp TIMESTAMP,
  load_date DATE,
  source_file_name STRING,
  record_id STRING,
  source_system STRING,
  is_deleted BOOLEAN,

  raw_data STRING,
  _rescued_data STRING
)

In [0]:
%sql
REPLACE TABLE retails.bronze.customers_raw
USING DELTA
PARTITIONED BY (load_date)
AS SELECT * FROM retails.bronze.customers_raw;

In [0]:
%sql
ALTER TABLE retails.bronze.inventory_raw
ADD COLUMNS(
  last_updated STRING,
  product_id STRING,
  stock_qty STRING,
  store_id STRING,
-- Metadata columns
  ingestion_timestamp TIMESTAMP,
  load_date DATE,
  source_file_name STRING,
  record_id STRING,
  source_system STRING,
  is_deleted BOOLEAN,

  raw_data STRING,
  _rescued_data STRING
);
    
REPLACE TABLE retails.bronze.inventory_raw
USING DELTA
PARTITIONED BY (load_date)
AS SELECT * FROM retails.bronze.inventory_raw;

In [0]:
products_df = spark.read \
    .format("json") \
    .option("mode", "DROPMALFORMED") \
    .option("columnNameOfCorruptRecord", "_corrupt_record") \
    .load("/Volumes/data/raw/sales/raw_data/retail/dev/products/")

# display(products_df);
products_df.printSchema()


In [0]:
%sql
ALTER TABLE retails.bronze.products_raw
ADD COLUMNS(
  category STRING,
  price STRING,
  product_id STRING,
  product_name STRING,
-- Metadata columns
  ingestion_timestamp TIMESTAMP,
  load_date DATE,
  source_file_name STRING,
  record_id STRING,
  source_system STRING,
  is_deleted BOOLEAN,

  raw_data STRING,
  _rescued_data STRING
);
    
REPLACE TABLE retails.bronze.products_raw
USING DELTA
PARTITIONED BY (load_date)
AS SELECT * FROM retails.bronze.products_raw;

In [0]:
stores_df = spark.read \
    .format("json") \
    .option("mode", "DROPMALFORMED") \
    .option("columnNameOfCorruptRecord", "_corrupt_record") \
    .load("/Volumes/data/raw/sales/raw_data/retail/dev/stores/")

# display(stores_df);
stores_df.printSchema()

In [0]:
%sql
ALTER TABLE retails.bronze.stores_raw
ADD COLUMNS(
  city STRING,
  state STRING,
  store_id STRING,
-- Metadata columns
  ingestion_timestamp TIMESTAMP,
  load_date DATE,
  source_file_name STRING,
  record_id STRING,
  source_system STRING,
  is_deleted BOOLEAN,

  raw_data STRING,
  _rescued_data STRING
);
    
REPLACE TABLE retails.bronze.stores_raw
USING DELTA
PARTITIONED BY (load_date)
AS SELECT * FROM retails.bronze.stores_raw;

In [0]:
%sql
describe table extended retails.bronze.pos_transactions_raw;

In [0]:
%sql
show grants on catalog retails;

In [0]:
%sql
show schemas in retails;

In [0]:
%sql
describe schema extended retails.bronze;

In [0]:
%sql
show tables in retails.bronze;

In [0]:
%sql
show grants on table retails.bronze.pos_transactions_raw;

In [0]:
%sql
show grants on schema retails.bronze;

In [0]:
%sql
show grants on schema retails.gold;

In [0]:
%sql
show catalogs;

In [0]:
dbutils.fs.ls("dbfs:/Volumes/data/raw/_schemas/retails/dev/customers/_schemas/")
# dbutils.fs.ls("dbfs:/Volumes/data/raw/_schemas/retails/dev/");



In [0]:
dbutils.fs.ls("dbfs:/Volumes/data/raw/_checkpoints/retails/dev/customers")

In [0]:
dbutils.fs.rm("dbfs:/Volumes/data/raw/_checkpoints/retails/dev/customers/", True)

In [0]:
%sql
show tables in retails.bronze;
-- drop table retails.bronze.customers12;

In [0]:
# creating checkpoint locations for auto loader for retails
dbutils.fs.mkdirs("dbfs:/Volumes/data/raw/_checkpoints/retails/dev/customers")
dbutils.fs.mkdirs("dbfs:/Volumes/data/raw/_checkpoints/retails/dev/inventory")
dbutils.fs.mkdirs("dbfs:/Volumes/data/raw/_checkpoints/retails/dev/pos_transactions")
dbutils.fs.mkdirs("dbfs:/Volumes/data/raw/_checkpoints/retails/dev/products")
dbutils.fs.mkdirs("dbfs:/Volumes/data/raw/_checkpoints/retails/dev/stores")

# creating schema locations  for auto loader for retails
dbutils.fs.mkdirs("dbfs:/Volumes/data/raw/_schemas/retails/dev/customers")
dbutils.fs.mkdirs("dbfs:/Volumes/data/raw/_schemas/retails/dev/inventory")
dbutils.fs.mkdirs("dbfs:/Volumes/data/raw/_schemas/retails/dev/pos_transactions")
dbutils.fs.mkdirs("dbfs:/Volumes/data/raw/_schemas/retails/dev/products")
dbutils.fs.mkdirs("dbfs:/Volumes/data/raw/_schemas/retails/dev/stores")

# dbutils.fs.rm("dbfs:/Volumes/data/raw/_checkpoints/test", True)

In [0]:
%sql
show volumes in data.raw;

In [0]:
%sql
describe history retails.bronze.inventory_raw;

In [0]:
# dbutils.fs.rm("dbfs:/Volumes/data/raw/_checkpoints/retails/dev/bronze/customers/", True)
# dbutils.fs.rm("dbfs:/Volumes/data/raw/_schemas/retails/dev/customers/", True)

# dbutils.fs.ls("dbfs:/Volumes/data/raw/_checkpoints/retails/dev/stores")
# dbutils.fs.ls("dbfs:/Volumes/data/raw/_schemas/retails/dev/stores")


In [0]:
%sql
-- DROP TABLE retails.bronze.inventory_raw;
-- DROP TABLE retails.bronze.products_raw;
-- DROP TABLE retails.bronze.customers_raw;
-- DROP TABLE retails.bronze.stores_raw;
-- DROP TABLE retails.bronze.pos_transactions_raw;

In [0]:
pos_df = spark.read.format("json") \
      .load("dbfs:/Volumes/data/raw/sales/raw_data/retail/dev/pos_transactions/")

pos_df.display()

In [0]:
from pyspark.sql.functions import col
# pos_df.select(pos_df.date.alias("transaction_date")).distinct().limit(10).display()
pos_df.withColumnRenamed("date", "txn_date").select("txn_date").distinct().limit(10).display()

In [0]:
from pyspark.sql.functions import col
# display(pos_df.select("_corrupt_record", "order_id").filter(col("_corrupt_record").isNotNull()).count())
# pos_df.select("_corrupt_record", "order_id").filter(col("_corrupt_record").isNull()).count()

# pos_df.select("_corrupt_record", "order_id") \
#       .filter(col("_corrupt_record").isNotNull()) \
#       .count()

# pos_df_cached = pos_df.cache()
# pos_df_cached.filter(col("_corrupt_record").isNotNull()).count()

# pos_df = pos_df.persist()
# display(pos_df.filter(col("_corrupt_record").isNotNull()).count())

pos_df.createOrReplaceTempView("pos")

spark.sql("""
SELECT COUNT(*) 
FROM pos 
WHERE _corrupt_record IS NOT NULL
""").show()

In [0]:
# remove delta logs
dbutils.fs.rm("s3://thoughtbulls-dp-uc-root-ap-south-1-8affd0fb/external_data/bronze/retails/pos_transactions_raw/", True)

In [0]:
customers_df = spark.read.table("retails.bronze.customers_raw")
# print(customers_df.columns)
customers_df.printSchema()


In [0]:
print(spark.version)

In [0]:
from pyspark.sql.functions import col

spark.table("retails.bronze.inventory_raw").filter(col("ingestion_dt") > "2026-05-02").display()

In [0]:
spark.read \
    .format("csv") \
    .schema("product_id string, store_id string") \
    .load("dbfs:/Volumes/data/raw/inventory/raw_data/retail/dev/inventory/")